# CNN-GNN-Transformer Maneuver Classifier

This notebook replaces the temporal LSTM with a lightweight transformer encoder over fused timestep embeddings.


## Run Safety

Before running this notebook, restart the kernel.

Why this matters:
- previous model runs can leave tensors and cached arrays in memory
- old variables can leak into the current run and make comparisons unfair
- we want every notebook to save a clean, reproducible result

Recommended workflow:
1. Restart kernel
2. Run all cells from top to bottom
3. Let the notebook save its outputs before closing it


In [1]:
# Memory cleanup before starting this notebook.
import gc

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

print('Kernel memory cleanup complete. Start the notebook from here after a restart.')


Kernel memory cleanup complete. Start the notebook from here after a restart.


In [2]:
# Standard-library imports used throughout the evaluation notebooks.
import json
import math
import random
import sys
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from datetime import datetime

# Core scientific / ML stack.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

# ROC / PR curves are useful for classification comparison if sklearn is available.
try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    from sklearn.preprocessing import label_binarize
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Resolve the project root robustly no matter where Jupyter was launched from.
SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = SEARCH_ROOT
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / '04_model_evaluation').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_ROOT = PROJECT_ROOT / '04_model_evaluation'
THESIS_ROOT = EVAL_ROOT
NOTEBOOK_ROOT = EVAL_ROOT / 'notebooks'
RESULTS_ROOT = EVAL_ROOT / 'results'
WEIGHTS_ROOT = EVAL_ROOT / 'model_weights'
SPLITS_ROOT = EVAL_ROOT / 'splits'
PLOTS_ROOT = EVAL_ROOT / 'plots'
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))
ORIGINAL_DATASET_ROOT = PROJECT_ROOT / '03_dataset' / 'hybrid_maneuver_dataset'
PREFERRED_DATASET_ROOT = Path('/media/basudeo/1044063744061FD8/hybrid_maneuver_dataset')
DATASET_ROOT = PREFERRED_DATASET_ROOT if PREFERRED_DATASET_ROOT.exists() else ORIGINAL_DATASET_ROOT

print('Project root:', PROJECT_ROOT)
print('Evaluation workspace:', THESIS_ROOT)
print('Original dataset root:', ORIGINAL_DATASET_ROOT)
print('Preferred external dataset root:', PREFERRED_DATASET_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Torch version:', torch.__version__)
print('Sklearn available for ROC/PR plots:', SKLEARN_AVAILABLE)


Project root: /home/basudeo/Documents/Thesis
Thesis workspace: /home/basudeo/Documents/Thesis/Thesis_Repo
Original dataset root: /home/basudeo/Documents/Thesis/hybrid_maneuver_dataset
Preferred external dataset root: /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset
Dataset root: /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset
Torch version: 2.3.1+cu118
Sklearn available for ROC/PR plots: True


In [3]:
# Shared experiment configuration.
# Keep this block explicit and heavily commented because it is the main place
# you will tune once more data has been collected.
LABEL_MODE = 'reduced'      # Use the balanced 5-class setting by default.
SEED = 42                   # Fix the split and initialization for reproducibility.
PAST_LEN = 10               # Number of past timesteps used to predict the current maneuver.
FUTURE_LEN = 5             # Number of future steps used for trajectory targets (ADE/FDE/RMSE).
SCAN_BEAMS = 512            # Number of lidar beams after resampling.
RANGE_CLIP = 30.0           # Maximum lidar range used for normalization.
BATCH_SIZE = 32             # Batch size for trainable models.
EPOCHS = 30                 # Upper bound; early stopping will usually stop sooner.
EARLY_STOPPING_PATIENCE = 5 # Patience on validation macro-F1.
LR = 1e-3                   # Adam learning rate.
WEIGHT_DECAY = 1e-4         # Small regularization on trainable models.
TRAJ_LOSS_WEIGHT = 1.0     # Weight for future-trajectory regression alongside classification.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None          # Set to an integer for quick smoke tests.
USE_CPU = False             # Force CPU if needed.

# Hidden sizes for the current classification baselines.
CNN_HIDDEN = 96
GRAPH_HIDDEN = 96
FUSION_HIDDEN = 128
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
MSG_PASSES = 2
DROPOUT = 0.10
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

# Device selection is kept simple and transparent.
device = torch.device('cuda' if torch.cuda.is_available() and not USE_CPU else 'cpu')
print('Device:', device)


Device: cuda


In [4]:
# Shared data and evaluation utilities live in one helper module so that
# all notebooks reuse the same dataset, split, path-remapping, and metric logic.
from dataset_helper import (
    SKLEARN_AVAILABLE,
    build_class_weights,
    build_label_mapping,
    build_run_manifest,
    build_sample_table,
    canonical_agent_order,
    compute_classification_metrics_from_probs,
    compute_trajectory_metrics,
    configure_helper,
    discover_frame_files,
    edge_features_for_order,
    group_streams,
    load_npy_cached,
    node_feature,
    prepare_result_dirs,
    remap_dataset_path,
    resample_scan,
    save_confusion_matrix,
    save_history_plot,
    save_mean_step_error_plot,
    save_or_load_fixed_split,
    save_predictions_csv,
    save_roc_pr_curves,
    save_run_manifest,
    save_training_history,
    save_trajectory_overlay_plots,
    set_seed,
    timestamp_tag,
)

configure_helper(
    dataset_root=DATASET_ROOT,
    original_dataset_root=ORIGINAL_DATASET_ROOT,
    results_root=RESULTS_ROOT,
    weights_root=WEIGHTS_ROOT,
)


In [5]:
@dataclass
class ClassificationSample:
    scan_seq: torch.Tensor | None
    node_seq: torch.Tensor | None
    edge_seq: torch.Tensor | None
    label: torch.Tensor
    future_xy: torch.Tensor
    future_dt: torch.Tensor
    sample_id: str


class BaseClassificationDataset(Dataset):
    """Base dataset that turns JSONL streams into time windows.

    Subclasses decide which modalities they need from each window.
    """
    def __init__(self, streams, sample_table, label_to_idx, past_len, scan_beams, range_clip):
        self.streams = streams
        self.sample_table = sample_table
        self.label_to_idx = label_to_idx
        self.past_len = past_len
        self.scan_beams = scan_beams
        self.range_clip = range_clip

    def __len__(self):
        return len(self.sample_table)

    def _window(self, idx):
        meta = self.sample_table[idx]
        stream = self.streams[meta['stream_idx']]
        return stream[meta['start']: meta['start'] + self.past_len], meta


class ScanOnlyDataset(BaseClassificationDataset):
    def __getitem__(self, idx):
        window, meta = self._window(idx)
        scan_seq = []
        for frame in window:
            scan_ref = frame['modalities']['ego_planar_scan']
            scan = load_npy_cached(str(scan_ref['path']))
            scan_seq.append(resample_scan(scan, self.scan_beams, self.range_clip))
        label_idx = self.label_to_idx[meta['label']]
        return ClassificationSample(
            scan_seq=torch.tensor(np.stack(scan_seq, axis=0), dtype=torch.float32),
            node_seq=None,
            edge_seq=None,
            label=torch.tensor(label_idx, dtype=torch.long),
            future_xy=torch.tensor(meta['future_xy'], dtype=torch.float32),
            future_dt=torch.tensor(meta['future_dt'], dtype=torch.float32),
            sample_id=meta['sample_id'],
        )


class GraphOnlyDataset(BaseClassificationDataset):
    def __getitem__(self, idx):
        window, meta = self._window(idx)
        node_seq, edge_seq = [], []
        ego_id = meta['ego_id']
        order = canonical_agent_order(ego_id)
        for frame in window:
            agents = frame['agents']
            ego_state = agents[ego_id]['state']
            node_seq.append([node_feature(agents[agent_id], ego_state) for agent_id in order])
            edge_seq.append(edge_features_for_order(frame, order))
        label_idx = self.label_to_idx[meta['label']]
        return ClassificationSample(
            scan_seq=None,
            node_seq=torch.tensor(node_seq, dtype=torch.float32),
            edge_seq=torch.tensor(edge_seq, dtype=torch.float32),
            label=torch.tensor(label_idx, dtype=torch.long),
            future_xy=torch.tensor(meta['future_xy'], dtype=torch.float32),
            future_dt=torch.tensor(meta['future_dt'], dtype=torch.float32),
            sample_id=meta['sample_id'],
        )


class ScanGraphDataset(BaseClassificationDataset):
    def __getitem__(self, idx):
        window, meta = self._window(idx)
        scan_seq, node_seq, edge_seq = [], [], []
        ego_id = meta['ego_id']
        order = canonical_agent_order(ego_id)
        for frame in window:
            scan_ref = frame['modalities']['ego_planar_scan']
            scan = load_npy_cached(str(scan_ref['path']))
            scan_seq.append(resample_scan(scan, self.scan_beams, self.range_clip))
            agents = frame['agents']
            ego_state = agents[ego_id]['state']
            node_seq.append([node_feature(agents[agent_id], ego_state) for agent_id in order])
            edge_seq.append(edge_features_for_order(frame, order))
        label_idx = self.label_to_idx[meta['label']]
        return ClassificationSample(
            scan_seq=torch.tensor(np.stack(scan_seq, axis=0), dtype=torch.float32),
            node_seq=torch.tensor(node_seq, dtype=torch.float32),
            edge_seq=torch.tensor(edge_seq, dtype=torch.float32),
            label=torch.tensor(label_idx, dtype=torch.long),
            future_xy=torch.tensor(meta['future_xy'], dtype=torch.float32),
            future_dt=torch.tensor(meta['future_dt'], dtype=torch.float32),
            sample_id=meta['sample_id'],
        )


def collate_samples(batch):
    scan_seq = None if batch[0].scan_seq is None else torch.stack([sample.scan_seq for sample in batch], dim=0)
    node_seq = None if batch[0].node_seq is None else torch.stack([sample.node_seq for sample in batch], dim=0)
    edge_seq = None if batch[0].edge_seq is None else torch.stack([sample.edge_seq for sample in batch], dim=0)
    labels = torch.stack([sample.label for sample in batch], dim=0)
    future_xy = torch.stack([sample.future_xy for sample in batch], dim=0)
    future_dt = torch.stack([sample.future_dt for sample in batch], dim=0)
    sample_ids = [sample.sample_id for sample in batch]
    return scan_seq, node_seq, edge_seq, labels, future_xy, future_dt, sample_ids


In [6]:
class LidarCNNEncoder(nn.Module):
    """1D CNN encoder for the planar lidar scan."""
    def __init__(self, in_channels=2, hidden_dim=CNN_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, hidden_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


In [7]:
class GraphEncoder(nn.Module):
    """Simple message-passing graph encoder reused across GNN-based notebooks."""
    def __init__(self, node_dim=14, edge_dim=8, hidden_dim=GRAPH_HIDDEN, msg_passes=MSG_PASSES):
        super().__init__()
        self.msg_passes = msg_passes
        self.node_proj = nn.Sequential(
            nn.Linear(node_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.node_update = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, node_feats, edge_feats):
        h = self.node_proj(node_feats)
        for _ in range(self.msg_passes):
            src = h.unsqueeze(2).expand(-1, -1, h.size(1), -1)
            dst = h.unsqueeze(1).expand(-1, h.size(1), -1, -1)
            messages = self.edge_mlp(torch.cat([src, dst, edge_feats], dim=-1))
            agg = messages.sum(dim=1)
            h = self.node_update(torch.cat([h, agg], dim=-1))
        return h


In [8]:
class LearnablePositionalEncoding(nn.Module):
    """A small learnable positional encoding for short temporal sequences."""
    def __init__(self, d_model, max_len=64):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos, mean=0.0, std=0.02)

    def forward(self, x):
        return x + self.pos[:, :x.size(1)]


In [9]:
def evaluate_classifier(model, loader, device, labels):
    model.eval()
    all_probs, all_targets, all_sample_ids = [], [], []
    all_pred_future_xy, all_true_future_xy = [], []
    total_loss = 0.0
    total_count = 0
    criterion_cls = nn.CrossEntropyLoss()
    criterion_traj = nn.MSELoss()
    with torch.no_grad():
        for scan_seq, node_seq, edge_seq, labels_batch, future_xy_batch, _future_dt_batch, sample_ids in loader:
            if scan_seq is not None: scan_seq = scan_seq.to(device)
            if node_seq is not None: node_seq = node_seq.to(device)
            if edge_seq is not None: edge_seq = edge_seq.to(device)
            labels_batch = labels_batch.to(device)
            future_xy_batch = future_xy_batch.to(device)
            logits, pred_future_xy = model(scan_seq, node_seq, edge_seq)
            loss_cls = criterion_cls(logits, labels_batch)
            loss_traj = criterion_traj(pred_future_xy, future_xy_batch)
            loss = loss_cls + TRAJ_LOSS_WEIGHT * loss_traj
            probs = torch.softmax(logits, dim=1)
            total_loss += float(loss.item()) * labels_batch.size(0)
            total_count += int(labels_batch.size(0))
            all_probs.append(probs.cpu().numpy())
            all_targets.append(labels_batch.cpu().numpy())
            all_pred_future_xy.append(pred_future_xy.cpu().numpy())
            all_true_future_xy.append(future_xy_batch.cpu().numpy())
            all_sample_ids.extend(sample_ids)
    probabilities = np.concatenate(all_probs, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    pred_future_xy = np.concatenate(all_pred_future_xy, axis=0)
    true_future_xy = np.concatenate(all_true_future_xy, axis=0)
    metrics, preds, confusion = compute_classification_metrics_from_probs(probabilities, targets, labels)
    metrics.update(compute_trajectory_metrics(pred_future_xy, true_future_xy))
    metrics['loss'] = total_loss / max(total_count, 1)
    return {'metrics': metrics, 'probabilities': probabilities, 'targets': targets, 'preds': preds, 'confusion': confusion, 'sample_ids': all_sample_ids, 'pred_future_xy': pred_future_xy, 'true_future_xy': true_future_xy}


def train_classifier(model, train_loader, val_loader, class_weights, labels, model_slug, timestamp, split_path, save_weights=True):
    result_dir, weight_dir, plot_dir = prepare_result_dirs(model_slug)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion_cls = nn.CrossEntropyLoss(weight=class_weights.to(device))
    criterion_traj = nn.MSELoss()
    history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'val_macro_f1': [], 'val_ade': [], 'val_fde': [], 'val_rmse': []}
    best_val_f1 = -math.inf
    best_state = None
    epochs_without_improvement = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        running_count = 0
        for scan_seq, node_seq, edge_seq, labels_batch, future_xy_batch, _future_dt_batch, _sample_ids in train_loader:
            if scan_seq is not None: scan_seq = scan_seq.to(device)
            if node_seq is not None: node_seq = node_seq.to(device)
            if edge_seq is not None: edge_seq = edge_seq.to(device)
            labels_batch = labels_batch.to(device)
            future_xy_batch = future_xy_batch.to(device)
            optimizer.zero_grad()
            logits, pred_future_xy = model(scan_seq, node_seq, edge_seq)
            loss_cls = criterion_cls(logits, labels_batch)
            loss_traj = criterion_traj(pred_future_xy, future_xy_batch)
            loss = loss_cls + TRAJ_LOSS_WEIGHT * loss_traj
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += float(loss.item()) * labels_batch.size(0)
            running_count += int(labels_batch.size(0))
        train_loss = running_loss / max(running_count, 1)
        val_eval = evaluate_classifier(model, val_loader, device, labels)
        val_metrics = val_eval['metrics']
        history['epoch'].append(epoch); history['train_loss'].append(train_loss); history['val_loss'].append(val_metrics['loss']); history['val_accuracy'].append(val_metrics['accuracy']); history['val_macro_f1'].append(val_metrics['macro_f1']); history['val_ade'].append(val_metrics['ADE']); history['val_fde'].append(val_metrics['FDE']); history['val_rmse'].append(val_metrics['RMSE'])
        print(f"Epoch {epoch:02d}/{EPOCHS} train_loss={train_loss:.5f} val_loss={val_metrics['loss']:.5f} val_acc={val_metrics['accuracy']:.4f} val_f1={val_metrics['macro_f1']:.4f} val_ADE={val_metrics['ADE']:.4f}")
        if val_metrics['macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics['macro_f1']; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f'Early stopping at epoch {epoch:02d}')
                break
    if best_state is None: raise RuntimeError('Training ended without a valid best checkpoint.')
    model.load_state_dict(best_state)
    run_manifest = build_run_manifest(model_slug=model_slug, timestamp=timestamp, labels=labels, split_path=split_path, extra=MODEL_EXTRA_MANIFEST)
    manifest_path = save_run_manifest(result_dir, run_manifest, timestamp)
    weight_payload = {'model_state': best_state, 'labels': labels, 'timestamp': timestamp, 'model_slug': model_slug, 'run_manifest': run_manifest}
    if save_weights:
        torch.save(weight_payload, weight_dir / f'{model_slug}_{timestamp}.pt')
        torch.save(weight_payload, weight_dir / 'latest.pt')
    save_training_history(history, result_dir / f'history_{timestamp}.csv')
    save_history_plot(history, plot_dir / f'history_{timestamp}.png', model_slug)
    return {'history': history, 'best_val_f1': float(best_val_f1), 'result_dir': result_dir, 'plot_dir': plot_dir, 'weight_dir': weight_dir, 'run_manifest_path': manifest_path, 'run_manifest': run_manifest}


def save_final_evaluation(model_slug, timestamp, train_out, test_eval, labels, extra_summary=None, split_path=None):
    result_dir = train_out['result_dir'] if train_out is not None else prepare_result_dirs(model_slug)[0]
    plot_dir = train_out['plot_dir'] if train_out is not None else prepare_result_dirs(model_slug)[2]
    metrics = dict(test_eval['metrics'])
    if split_path is not None: metrics['split_path'] = str(split_path)
    if train_out is not None and 'best_val_f1' in train_out: metrics['best_val_macro_f1'] = float(train_out['best_val_f1'])
    if extra_summary: metrics.update(extra_summary)
    roc_summary = save_roc_pr_curves(test_eval['probabilities'], test_eval['targets'], labels, plot_dir)
    metrics.update(roc_summary)
    metrics_path = result_dir / f'metrics_{timestamp}.json'
    metrics_path.write_text(json.dumps(metrics, indent=2))
    (result_dir / 'latest_metrics.json').write_text(json.dumps(metrics, indent=2))
    if train_out is not None and 'run_manifest' in train_out: (result_dir / 'latest_run_manifest.json').write_text(json.dumps(train_out['run_manifest'], indent=2))
    save_predictions_csv(test_eval['sample_ids'], test_eval['targets'], test_eval['preds'], test_eval['probabilities'], labels, result_dir / f'predictions_{timestamp}.csv')
    save_confusion_matrix(test_eval['confusion'], labels, result_dir / f'confusion_{timestamp}.csv', plot_dir / f'confusion_{timestamp}.png', f'{model_slug} Confusion Matrix')
    overlay_paths = save_trajectory_overlay_plots(test_eval['pred_future_xy'], test_eval['true_future_xy'], test_eval['targets'], labels, plot_dir, prefix=timestamp, max_plots=8)
    step_error_path = save_mean_step_error_plot(test_eval['pred_future_xy'], test_eval['true_future_xy'], plot_dir / f'mean_step_error_{timestamp}.png', title=f'{model_slug} Mean Future-Step Error')
    metrics['trajectory_overlay_plots'] = overlay_paths
    metrics['mean_step_error_plot'] = step_error_path
    metrics_path.write_text(json.dumps(metrics, indent=2))
    (result_dir / 'latest_metrics.json').write_text(json.dumps(metrics, indent=2))
    return metrics

MODEL_EXTRA_MANIFEST = {'cnn_hidden': CNN_HIDDEN, 'graph_hidden': GRAPH_HIDDEN, 'fusion_hidden': FUSION_HIDDEN, 'transformer_heads': TRANSFORMER_HEADS, 'transformer_layers': TRANSFORMER_LAYERS, 'transformer_ff': TRANSFORMER_FF, 'dropout': DROPOUT, 'future_len': FUTURE_LEN, 'trajectory_loss_weight': TRAJ_LOSS_WEIGHT}


In [10]:
set_seed(SEED)
labels, label_mapping = build_label_mapping(LABEL_MODE)
label_to_idx = {label: idx for idx, label in enumerate(labels)}
streams = group_streams(DATASET_ROOT, allowed_labels=set(labels), label_mapping=label_mapping)
sample_table = build_sample_table(streams, past_len=PAST_LEN, future_len=FUTURE_LEN)
split_path = SPLITS_ROOT / f'classification_split_{LABEL_MODE}_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json'
split_info = save_or_load_fixed_split(sample_table, split_path, SEED, TRAIN_RATIO, VAL_RATIO, PAST_LEN, FUTURE_LEN)

print('Labels:', labels)
print('Split path:', split_path)
print('Total samples in canonical table:', len(sample_table))
print('Train / Val / Test:', len(split_info['train_indices']), len(split_info['val_indices']), len(split_info['test_indices']))
print('Future horizon:', split_info['future_len'])


Labels: ['go_to_goal', 'avoid_left', 'avoid_right', 'commit_forward', 'arrived']
Split path: /home/basudeo/Documents/Thesis/Thesis_Repo/splits/classification_split_reduced_seed42_past10_future5.json
Total samples in canonical table: 11488
Train / Val / Test: 2844 609 610
Future horizon: 5


In [11]:
class LidarCNNEncoder(nn.Module):
    """1D CNN encoder for the planar lidar scan."""
    def __init__(self, in_channels=2, hidden_dim=CNN_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, hidden_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

class GraphEncoder(nn.Module):
    """Simple message-passing graph encoder reused across GNN-based notebooks."""
    def __init__(self, node_dim=14, edge_dim=8, hidden_dim=GRAPH_HIDDEN, msg_passes=MSG_PASSES):
        super().__init__()
        self.msg_passes = msg_passes
        self.node_proj = nn.Sequential(
            nn.Linear(node_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.node_update = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, node_feats, edge_feats):
        h = self.node_proj(node_feats)
        for _ in range(self.msg_passes):
            src = h.unsqueeze(2).expand(-1, -1, h.size(1), -1)
            dst = h.unsqueeze(1).expand(-1, h.size(1), -1, -1)
            messages = self.edge_mlp(torch.cat([src, dst, edge_feats], dim=-1))
            agg = messages.sum(dim=1)
            h = self.node_update(torch.cat([h, agg], dim=-1))
        return h

class LearnablePositionalEncoding(nn.Module):
    """A small learnable positional encoding for short temporal sequences."""
    def __init__(self, d_model, max_len=64):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos, mean=0.0, std=0.02)

    def forward(self, x):
        return x + self.pos[:, :x.size(1)]

class CNNGNNTransformerMultiTask(nn.Module):
    """Transformer over fused CNN+GNN timestep embeddings."""
    def __init__(self, num_classes, future_len):
        super().__init__()
        self.future_len = future_len
        self.scan_encoder = LidarCNNEncoder(in_channels=2, hidden_dim=CNN_HIDDEN)
        self.graph_encoder = GraphEncoder(node_dim=14, edge_dim=8, hidden_dim=GRAPH_HIDDEN, msg_passes=MSG_PASSES)
        self.fusion = nn.Sequential(
            nn.Linear(CNN_HIDDEN + GRAPH_HIDDEN, FUSION_HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
        )
        self.pos = LearnablePositionalEncoding(FUSION_HIDDEN, max_len=64)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=FUSION_HIDDEN,
            nhead=TRANSFORMER_HEADS,
            dim_feedforward=TRANSFORMER_FF,
            dropout=DROPOUT,
            batch_first=True,
            activation='relu',
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=TRANSFORMER_LAYERS)
        self.classifier = nn.Sequential(
            nn.Linear(FUSION_HIDDEN, FUSION_HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FUSION_HIDDEN, num_classes),
        )
        self.traj_head = nn.Sequential(
            nn.Linear(FUSION_HIDDEN, FUSION_HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FUSION_HIDDEN, future_len * 2),
        )

    def forward(self, scan_seq, node_seq, edge_seq):
        fused_steps = []
        for t in range(scan_seq.size(1)):
            scan_emb = self.scan_encoder(scan_seq[:, t])
            graph_emb = self.graph_encoder(node_seq[:, t], edge_seq[:, t])[:, 0]
            fused_steps.append(self.fusion(torch.cat([scan_emb, graph_emb], dim=-1)))
        seq = torch.stack(fused_steps, dim=1)
        seq = self.pos(seq)
        encoded = self.transformer(seq)
        pooled = encoded.mean(dim=1)
        logits = self.classifier(pooled)
        future_xy = self.traj_head(pooled).view(pooled.size(0), self.future_len, 2)
        return logits, future_xy


In [12]:
# Build the dataset for this model family.
DATASET_CLASS = ScanGraphDataset
MODEL_SLUG = 'cnn_gnn_transformer'
TIMESTAMP = timestamp_tag()

dataset = DATASET_CLASS(
    streams=streams,
    sample_table=sample_table,
    label_to_idx=label_to_idx,
    past_len=PAST_LEN,
    scan_beams=SCAN_BEAMS,
    range_clip=RANGE_CLIP,
)

train_subset = Subset(dataset, split_info['train_indices'])
val_subset = Subset(dataset, split_info['val_indices'])
test_subset = Subset(dataset, split_info['test_indices'])

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_samples)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_samples)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_samples)

train_label_indices = [int(dataset[idx].label.item()) for idx in split_info['train_indices']]
class_weights = build_class_weights(train_label_indices, num_classes=len(labels))
print('Class weights:', class_weights.tolist())

model = CNNGNNTransformerMultiTask(num_classes=len(labels), future_len=FUTURE_LEN).to(device)
train_out = train_classifier(model, train_loader, val_loader, class_weights, labels, MODEL_SLUG, TIMESTAMP, split_path=split_path, save_weights=True)
test_eval = evaluate_classifier(model, test_loader, device, labels)
final_metrics = save_final_evaluation(MODEL_SLUG, TIMESTAMP, train_out, test_eval, labels, split_path=split_path)
print('Final test metrics:')
print(json.dumps(final_metrics, indent=2))


Class weights: [0.46508586406707764, 4.739999771118164, 5.469230651855469, 0.6434389352798462, 1.1087719202041626]
Epoch 01/30 train_loss=1.63658 val_loss=1.50067 val_acc=0.2512 val_f1=0.1718 val_ADE=0.3107
Epoch 02/30 train_loss=1.56664 val_loss=1.48073 val_acc=0.3924 val_f1=0.2836 val_ADE=0.3023
Epoch 03/30 train_loss=1.48500 val_loss=1.33769 val_acc=0.3842 val_f1=0.2942 val_ADE=0.2880
Epoch 04/30 train_loss=1.32170 val_loss=1.22226 val_acc=0.5829 val_f1=0.4219 val_ADE=0.2839
Epoch 05/30 train_loss=1.21217 val_loss=1.18226 val_acc=0.4713 val_f1=0.4200 val_ADE=0.2636
Epoch 06/30 train_loss=1.16271 val_loss=1.11062 val_acc=0.5731 val_f1=0.3994 val_ADE=0.2680
Epoch 07/30 train_loss=1.15148 val_loss=0.97480 val_acc=0.6404 val_f1=0.4757 val_ADE=0.2889
Epoch 08/30 train_loss=1.14792 val_loss=1.06246 val_acc=0.5452 val_f1=0.4154 val_ADE=0.2627
Epoch 09/30 train_loss=1.11432 val_loss=1.02964 val_acc=0.6322 val_f1=0.4640 val_ADE=0.2503
Epoch 10/30 train_loss=1.11345 val_loss=0.97108 val_acc=0